In [4]:
"""
==========================================================================
BLOCK 1: CORE INFRASTRUCTURE AND SYSTEM GLOBALS
==========================================================================
Imports libraries, defines global constants, discovers input files.
==========================================================================
"""

# import system modules
#
import os
import glob
import warnings

# import numerical and dataframe modules
#
import numpy as np
import pandas as pd

# import machine learning modules
#
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import mutual_info_regression, RFE
import lightgbm as lgb
from lightgbm import LGBMRegressor

# suppress pandas downcasting warnings
#
pd.set_option('future.no_silent_downcasting', True)

# suppress lightgbm and sklearn warnings
#
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", message=".*sklearn.utils.parallel.delayed.*")

# ---------------------------------------------------------------------------
#
# user-configurable paths: set these to match your local directory layout
#
# ---------------------------------------------------------------------------

# define the directory containing the raw indoor and outdoor csv files
#
BASE_PATH = "/home/dylan/proeject_data_science/raw/"

# define the output directory for processed daily files
#
OUT_DAILY = os.path.join(BASE_PATH, "processed_daily")

# ---------------------------------------------------------------------------
#
# pipeline constants
#
# ---------------------------------------------------------------------------

# define the time threshold in minutes for gap detection and interpolation
#
TIME_THRESHOLD_MINUTES = 30

# define the numeric token used to terminate setpoint forward-fill
#
NUMERIC_RESET_TOKEN = -9999.9

# define step-like columns that get forward-filled between virtual rows
#
STEP_LIKE_COLS = [
    "Setpoint", "FanState"
]

# define columns excluded from feature selection entirely.
# daily_fan_on_hours and fan_runtime_ratio are perfect proxies for the
# target (fan=on agrees with running_mode=heat/cool 100% of the time).
# total_included_known_mode_hours and total_unknown_hours are data
# quality metrics that encode reporting gaps, not thermal behavior.
#
LEAKAGE_COLS = [
    "daily_runtime_hours",
    "daily_fan_on_hours",
    "fan_runtime_ratio"
]

# define the number of lag days for historical memory features
#
LAG_DAYS = 7

# define the window size for outdoor temperature trend gradient
#
OUTDOOR_TREND_WINDOW = 7

# create the output directory if it does not exist
#
os.makedirs(OUT_DAILY, exist_ok=True)

# discover all csv files in the base directory
#
all_raw = sorted(glob.glob(os.path.join(BASE_PATH, "*.csv")))

# identify the outdoor weather file by name
#
outdoor_file = [f for f in all_raw if "outdoor" in os.path.basename(f).lower()]

# verify exactly one outdoor file was found
#
if len(outdoor_file) != 1:
    raise FileNotFoundError(
        f"expected 1 outdoor file, found {len(outdoor_file)}: {outdoor_file}"
    )

# extract the outdoor file path
#
outdoor_file = outdoor_file[0]

# collect all remaining csv files as indoor thermostat sources,
# excluding any previously processed daily output files
#
indoor_files = [
    f for f in all_raw
    if f != outdoor_file and not os.path.basename(f).endswith("_daily.csv")
]

# print file discovery results
#
print(f"outdoor file: {os.path.basename(outdoor_file)}")
print(f"indoor files: {len(indoor_files)}")


outdoor file: outdoorweather.csv
indoor files: 100


In [ ]:
"""
==========================================================================
BLOCK 2: OUTDOOR WEATHER PREPARATION
==========================================================================
Ingests and standardizes the outdoor weather file. Ensures temperature
and humidity columns are numeric and chronologically sorted.
==========================================================================
"""

def process_outdoor_weather(path):
    """
    function: process_outdoor_weather

    arguments:
      path: file path to the outdoor weather csv

    return:
      dataframe with Timestamp and four outdoor variables

    description:
      reads, cleans, and standardizes the outdoor weather file.
    """

    # read the outdoor file with semicolon delimiter
    #
    dfw = pd.read_csv(path, sep=";", low_memory=False)

    # parse the timestamp column
    #
    dfw["Timestamp"] = pd.to_datetime(dfw["Timestamp"], errors="coerce")

    # drop rows with invalid timestamps and sort chronologically
    #
    dfw = dfw.dropna(subset=["Timestamp"]).sort_values("Timestamp").reset_index(drop=True)

    # rename Temperature to Outdoor_Temperature to avoid indoor collision
    #
    dfw = dfw.rename(columns={"Temperature": "Outdoor_Temperature"})

    # define the required outdoor columns
    #
    target_cols = [
        "Outdoor_Temperature", "outsideMinTemp",
        "outsideMaxTemp", "outsideHumidity"
    ]

    # coerce each target column to numeric
    #
    for c in target_cols:
        dfw[c] = pd.to_numeric(dfw[c], errors="coerce")

    # return only the timestamp and target columns
    #
    return dfw[["Timestamp"] + target_cols]

# process the outdoor weather file
#
w_df = process_outdoor_weather(outdoor_file)

# confirm outdoor weather processing
#
print("\noutdoor weather processed:")
print(w_df.head())


In [ ]:
"""
==========================================================================
BLOCK 3: INDOOR FILE INGESTION AND COLUMN STANDARDIZATION
==========================================================================
Maps raw manufacturer column names to canonical names. Parses timestamps,
adds Equipment_ID, and creates a raw row counter.
==========================================================================
"""

def standardize_indoor_columns(df):
    """
    function: standardize_indoor_columns

    arguments:
      df: raw indoor dataframe

    return:
      dataframe with canonical column names

    description:
      renames known raw column names to standard pipeline names.
    """

    # define the column rename mapping
    #
    rename_map = {
        "output_state": "OutputState",
        "fan_state": "FanState",
        "running_mode": "RunningMode",
        "hvac_state": "RunningMode",
        "temp": "Temperature",
        "set_point": "Setpoint"
    }

    # apply the rename map
    #
    df = df.rename(columns=rename_map)

    # return the renamed dataframe
    #
    return df

# process the first indoor file as a sample walkthrough
#
sample_path = indoor_files[0]

# read the raw sample file
#
sample_df = pd.read_csv(sample_path, sep=";", low_memory=False)

# standardize column names
#
sample_df = standardize_indoor_columns(sample_df)

# parse the timestamp column
#
sample_df["Timestamp"] = pd.to_datetime(sample_df["Timestamp"], errors="coerce")

# drop invalid timestamps and sort
#
sample_df = sample_df.dropna(subset=["Timestamp"]).sort_values("Timestamp").reset_index(drop=True)

# extract the equipment id from the filename
#
sample_df["Equipment_ID"] = os.path.splitext(os.path.basename(sample_path))[0]

# add a row counter to preserve ordering on timestamp ties
#
sample_df["raw_row_id"] = np.arange(len(sample_df))

# confirm ingestion
#
print(f"\nraw ingestion for {sample_df['Equipment_ID'].iloc[0]}:")
print(sample_df[["Timestamp", "Equipment_ID", "RunningMode", "Temperature", "Setpoint"]].head())


In [ ]:
"""
==========================================================================
BLOCK 4: BURST RESOLUTION (SAME-TIMESTAMP SNAPSHOTS)
==========================================================================
Resolves cases where the device sent multiple rows at the exact same
timestamp. Forward-fills within each burst and keeps the last row.
==========================================================================
"""

def resolve_same_timestamp_bursts(df):
    """
    function: resolve_same_timestamp_bursts

    arguments:
      df: indoor event dataframe

    return:
      dataframe with one row per unique timestamp

    description:
      forward-fills within each timestamp group, keeps the last row.
    """

    # isolate all non-timestamp columns
    #
    data_cols = [c for c in df.columns if c != "Timestamp"]

    # forward-fill within each timestamp group
    #
    df[data_cols] = df.groupby("Timestamp", sort=False)[data_cols].ffill()

    # keep only the last row of each timestamp group
    #
    last_rows = df.groupby("Timestamp", sort=False, as_index=False).tail(1)

    # reset the index
    #
    return last_rows.reset_index(drop=True)

# resolve bursts in the sample
#
sample_df = resolve_same_timestamp_bursts(sample_df)

# confirm burst resolution
#
print("\nburst resolution complete:")
print(sample_df[["Timestamp", "raw_row_id", "RunningMode", "Temperature"]].head())


In [ ]:
"""
==========================================================================
BLOCK 5: STATE NORMALIZATION AND VIRTUAL EXPIRATION ROW INJECTION
==========================================================================
Normalizes running mode text to canonical states (heat/cool/off/unknown).
Detects gaps exceeding 30 minutes and injects synthetic expiration rows
to terminate forward-fill persistence.
==========================================================================
"""

def normalize_running_mode(x):
    """
    function: normalize_running_mode

    arguments:
      x: raw running mode value

    return:
      canonical mode string or np.nan

    description:
      maps raw mode text to heat, cool, off, or unknown.
    """

    # preserve nulls
    #
    if pd.isna(x):
        return np.nan

    # lowercase and strip whitespace
    #
    s = str(x).strip().lower()

    # map to heating
    #
    if s in {"heat", "heating"}:
        return "heat"

    # map to cooling
    #
    if s in {"cool", "cooling"}:
        return "cool"

    # map to off/idle
    #
    if s in {"off", "idle", "none", "false", "0"}:
        return "off"

    # default to unknown
    #
    return "unknown"


def inject_virtual_expiration_rows(df):
    """
    function: inject_virtual_expiration_rows

    arguments:
      df: indoor event dataframe

    return:
      dataframe with synthetic expiration rows inserted

    description:
      for each gap > 30 min, inserts a synthetic row at gap_start + 30 min
      to reset step-like states.
    """

    # compute the time gap in seconds between every consecutive pair of
    # rows. a large gap indicates the thermostat stopped sending all
    # pings (a true data blackout, not just a missing column).
    #
    gap_sec = df["Timestamp"].diff().dt.total_seconds()

    # identify rows where the preceding gap exceeds the 30-minute
    # threshold. these are the points where we need to inject a
    # synthetic expiration row to terminate forward-fill persistence.
    #
    blackout_mask = gap_sec > (TIME_THRESHOLD_MINUTES * 60)

    # if no blackouts were found, return the dataframe unchanged
    #
    if not blackout_mask.any():
        return df

    # compute the timestamp where each blackout began by subtracting
    # the gap duration from the row that follows the gap. the virtual
    # row is placed 30 minutes after that gap-start timestamp.
    #
    bl_ts = df.loc[blackout_mask, "Timestamp"]
    bl_gap = gap_sec[blackout_mask]
    gap_starts = bl_ts - pd.to_timedelta(bl_gap, unit="s")
    virtual_timestamps = gap_starts.values + pd.Timedelta(minutes=TIME_THRESHOLD_MINUTES)

    # build all virtual rows at once as a single dataframe instead of
    # appending one dict at a time. each virtual row resets the
    # step-like columns (running mode, setpoint, fan state) to
    # terminate persistence so forward-fill does not bleed state
    # across the data blackout.
    #
    v_rows = pd.DataFrame({
        "Timestamp": virtual_timestamps,
        "Equipment_ID": df.loc[blackout_mask, "Equipment_ID"].values,
        "RunningMode_clean": "unknown",
        "Setpoint": NUMERIC_RESET_TOKEN,
        "FanState": "unknown",
        "is_virtual_expiration": True,
    })

    # concatenate the virtual rows with the original dataframe and
    # re-sort so the synthetic rows sit in their correct temporal
    # positions within the timeline
    #
    df = pd.concat([df, v_rows], ignore_index=True)
    df = df.sort_values("Timestamp").reset_index(drop=True)

    # return the augmented dataframe
    #
    return df

# copy the raw running mode for auditing
#
sample_df["RunningMode_raw"] = sample_df.get("RunningMode", np.nan)

# normalize running modes
#
sample_df["RunningMode_clean"] = sample_df["RunningMode_raw"].map(normalize_running_mode)

# inject virtual expiration rows
#
sample_df = inject_virtual_expiration_rows(sample_df)

# confirm virtual row injection
#
virt = sample_df[sample_df.get('is_virtual_expiration') == True]
print(f"\nvirtual expiration rows injected: {len(virt)}")
print(virt[["Timestamp", "RunningMode_clean", "Setpoint", "FanState"]].head())


In [ ]:
"""
==========================================================================
BLOCK 6: UNBOUNDED STATE PERSISTENCE AND BOUNDED INTERPOLATION
==========================================================================
Forward-fills step-like columns (fill stops at virtual rows). Applies
bounded time interpolation to indoor temperature: only keeps interpolated
values where the total gap between valid readings is <= 30 minutes.
==========================================================================
"""

def bounded_time_interpolate(series, times):
    """
    function: bounded_time_interpolate

    arguments:
      series: pandas Series of values to interpolate
      times: pandas Series of corresponding timestamps

    return:
      pandas Series with bounded interpolated values

    description:
      interpolates using time index, but nullifies any interpolated
      value where the surrounding valid-to-valid gap exceeds 30 min.
    """

    # coerce to numeric
    #
    s = pd.to_numeric(series, errors="coerce")

    # find timestamps of previous and next valid values
    #
    prev_valid_ts = times.where(s.notna()).ffill()
    next_valid_ts = times.where(s.notna()).bfill()

    # compute total gap width in minutes
    #
    total_gap_min = (next_valid_ts - prev_valid_ts).dt.total_seconds() / 60.0

    # perform time-based interpolation
    #
    temp = pd.DataFrame({"t": times, "v": s}).set_index("t")
    interp_vals = temp["v"].interpolate(method="time").values

    # keep values only where original was valid or gap <= threshold
    #
    ok_mask = s.notna() | (total_gap_min <= TIME_THRESHOLD_MINUTES)

    # return interpolated values with invalid regions nullified
    #
    return pd.Series(np.where(ok_mask, interp_vals, np.nan), index=series.index)

# forward-fill the normalized running mode
#
sample_df["RunningMode"] = sample_df["RunningMode_clean"].ffill()

# apply bounded interpolation to indoor temperature
#
sample_df["Temperature"] = bounded_time_interpolate(
    sample_df["Temperature"], sample_df["Timestamp"]
)

# forward-fill and back-fill static metadata if present
#
if "RentalStatus" in sample_df.columns:
    sample_df["RentalStatus"] = sample_df["RentalStatus"].ffill().bfill()

# forward-fill each step-like column
#
for c in STEP_LIKE_COLS:
    if c in sample_df.columns:

        # coerce setpoint to numeric before filling
        #
        if c == "Setpoint":
            sample_df[c] = pd.to_numeric(sample_df[c], errors="coerce")

        # forward-fill the column
        #
        sample_df[c] = sample_df[c].ffill()

        # replace the reset token with NaN for setpoint
        #
        if c == "Setpoint":
            sample_df[c] = sample_df[c].replace(NUMERIC_RESET_TOKEN, np.nan)

# re-sort by timestamp and raw row id
#
sample_df = sample_df.sort_values(
    ["Timestamp", "raw_row_id"], kind="mergesort"
).reset_index(drop=True)

# confirm fill and interpolation
#
print("\nstate persistence and interpolation complete:")
print(sample_df[["Timestamp", "RunningMode", "Temperature", "Setpoint"]].head(10))


In [ ]:
"""
==========================================================================
BLOCK 7: INTERVAL RUNTIME ACCOUNTING
==========================================================================
Computes the duration in seconds between consecutive events and assigns
each interval to the prior operating mode.
==========================================================================
"""

# compute the previous row timestamp
#
sample_df["prev_timestamp"] = sample_df["Timestamp"].shift(1)

# compute interval duration in seconds
#
sample_df["Duration_Seconds"] = (
    sample_df["Timestamp"] - sample_df["prev_timestamp"]
).dt.total_seconds().fillna(0)

# get the mode active during each interval (from the previous row)
#
sample_df["Interval_RunningMode"] = sample_df["RunningMode"].shift(1).fillna("unknown")

# assign duration to mode-specific columns
#
event_dt = sample_df["Duration_Seconds"]
sample_df["heat_sec"] = np.where(sample_df["Interval_RunningMode"] == "heat", event_dt, 0.0)
sample_df["cool_sec"] = np.where(sample_df["Interval_RunningMode"] == "cool", event_dt, 0.0)
sample_df["off_sec"] = np.where(sample_df["Interval_RunningMode"] == "off", event_dt, 0.0)
sample_df["unk_sec"] = np.where(sample_df["Interval_RunningMode"] == "unknown", event_dt, 0.0)

# confirm interval accounting
#
print("\ninterval accounting:")
print(sample_df[["Timestamp", "Duration_Seconds", "Interval_RunningMode",
                  "heat_sec", "cool_sec", "unk_sec"]].head(10))


In [ ]:
"""
==========================================================================
BLOCK 8: OUTDOOR WEATHER ALIGNMENT
==========================================================================
Aligns outdoor readings to each indoor timestamp using merge_asof.
Interpolates outdoor temperature and humidity when the outdoor gap is
<= 30 minutes, otherwise sets to NaN.
==========================================================================
"""

def attach_weather_to_indoor(indoor_df, outdoor_df):
    """
    function: attach_weather_to_indoor

    arguments:
      indoor_df: indoor event dataframe
      outdoor_df: processed outdoor weather dataframe

    return:
      indoor_df with outdoor columns attached

    description:
      uses merge_asof to find previous and next outdoor readings for
      each indoor timestamp, then linearly interpolates within 30 min.
    """

    # extract sorted indoor timestamps
    #
    indoor_ts = indoor_df[["Timestamp"]].copy().sort_values("Timestamp")

    # find the most recent outdoor reading at or before each indoor timestamp
    #
    prev_w = pd.merge_asof(
        indoor_ts,
        outdoor_df[["Timestamp", "Outdoor_Temperature", "outsideHumidity"]].rename(
            columns={
                "Timestamp": "prev_outdoor_ts",
                "Outdoor_Temperature": "prev_outdoor_temp",
                "outsideHumidity": "prev_outdoor_humidity"
            }
        ),
        left_on="Timestamp", right_on="prev_outdoor_ts", direction="backward"
    )

    # find the next outdoor reading at or after each indoor timestamp
    #
    next_w = pd.merge_asof(
        indoor_ts,
        outdoor_df.rename(
            columns={
                "Timestamp": "next_outdoor_ts",
                "Outdoor_Temperature": "next_outdoor_temp",
                "outsideHumidity": "next_outdoor_humidity"
            }
        ),
        left_on="Timestamp", right_on="next_outdoor_ts", direction="forward"
    )

    # compute total outdoor gap in minutes
    #
    total_gap_min = (
        next_w["next_outdoor_ts"] - prev_w["prev_outdoor_ts"]
    ).dt.total_seconds() / 60.0

    # determine which rows have acceptable outdoor gaps
    #
    ok_mask = total_gap_min <= TIME_THRESHOLD_MINUTES

    # compute fractional interpolation weight
    #
    weight = (
        indoor_ts["Timestamp"] - prev_w["prev_outdoor_ts"]
    ).dt.total_seconds() / (total_gap_min * 60.0)

    # handle exact matches and missing values
    #
    weight = weight.replace([np.inf, -np.inf], np.nan).fillna(0)

    # interpolate outdoor temperature
    #
    ext_temp = np.where(
        ok_mask,
        prev_w["prev_outdoor_temp"] * (1 - weight) + next_w["next_outdoor_temp"] * weight,
        np.nan
    )

    # interpolate outdoor humidity
    #
    ext_hum = np.where(
        ok_mask,
        prev_w["prev_outdoor_humidity"] * (1 - weight) + next_w["next_outdoor_humidity"] * weight,
        np.nan
    )

    # attach interpolated outdoor columns to the indoor dataframe
    #
    indoor_df["Outdoor_Temperature"] = ext_temp
    indoor_df["outsideHumidity"] = ext_hum

    # attach min/max from the forward weather row
    #
    indoor_df["outsideMinTemp"] = next_w["outsideMinTemp"]
    indoor_df["outsideMaxTemp"] = next_w["outsideMaxTemp"]

    # return the enriched indoor dataframe
    #
    return indoor_df

# attach weather to the sample
#
sample_df = attach_weather_to_indoor(sample_df, w_df)

# confirm weather alignment
#
print("\nweather alignment complete:")
print(sample_df[["Timestamp", "Outdoor_Temperature", "outsideHumidity", "outsideMinTemp"]].head(10))


In [ ]:
"""
==========================================================================
BLOCK 9: DAILY AGGREGATION, TIME-WEIGHTED STATISTICS, AND OUTDOOR
         TEMPERATURE GRADIENT
==========================================================================
Slices event intervals at midnight boundaries. Aggregates to daily rows
using time-weighted statistics for all continuous variables (temperature,
setpoint, outdoor temperature, outdoor humidity). The same duration
weighting used for weighted means is also applied to variance, std,
skewness, kurtosis, and quantiles. Also computes the true daily outdoor
weather aggregates and the outdoor temperature trend gradient (linear
slope over prior N days).
==========================================================================
"""

def weighted_stats_for_slices(values, weights, prefix):
    """
    function: weighted_stats_for_slices

    arguments:
      values: array-like of numeric values
      weights: array-like of duration weights (seconds)
      prefix: string prefix for output dictionary keys

    return:
      dictionary of time-weighted distributional statistics

    description:
      computes time-weighted mean, variance, std, skewness, kurtosis,
      and approximate weighted quantiles using the same duration
      weighting as the weighted average.
    """

    # define all statistic keys
    #
    keys = [
        "min", "q25", "median", "q75", "max", "range",
        "mean", "std", "variance", "iqr",
        "skewness", "kurtosis_excess",
        "raw_moment_2", "raw_moment_3"
    ]

    # initialize output with nan defaults
    #
    out = {f"{prefix}_{k}": np.nan for k in keys}

    # coerce to numpy arrays and filter out nans
    #
    v = pd.to_numeric(pd.Series(values), errors="coerce").values
    w = np.asarray(weights, dtype=float)
    mask = np.isfinite(v) & np.isfinite(w) & (w > 0)
    v = v[mask]
    w = w[mask]

    # record valid count
    #
    n = len(v)
    out[f"{prefix}_count_nonnull"] = n

    # return early if no valid data
    #
    if n == 0:
        return out

    # compute total weight
    #
    w_sum = w.sum()

    # compute weighted mean
    #
    w_mean = np.dot(v, w) / w_sum

    # compute weighted min and max
    #
    out[f"{prefix}_min"] = v.min()
    out[f"{prefix}_max"] = v.max()
    out[f"{prefix}_range"] = v.max() - v.min()
    out[f"{prefix}_mean"] = w_mean

    # compute weighted raw moments
    #
    out[f"{prefix}_raw_moment_2"] = np.dot(v ** 2, w) / w_sum
    out[f"{prefix}_raw_moment_3"] = np.dot(v ** 3, w) / w_sum

    # compute weighted quantiles using sorted cumulative weights
    #
    sort_idx = np.argsort(v)
    v_sorted = v[sort_idx]
    w_sorted = w[sort_idx]
    cum_w = np.cumsum(w_sorted)
    cum_w_norm = (cum_w - 0.5 * w_sorted) / w_sum

    # interpolate quantile values
    #
    for q_val, q_name in [(0.25, "q25"), (0.50, "median"), (0.75, "q75")]:
        out[f"{prefix}_{q_name}"] = np.interp(q_val, cum_w_norm, v_sorted)

    # compute weighted iqr
    #
    out[f"{prefix}_iqr"] = out[f"{prefix}_q75"] - out[f"{prefix}_q25"]

    # compute weighted variance and std (require at least 2 points)
    #
    if n >= 2:

        # compute deviations from the weighted mean
        #
        deviations = v - w_mean

        # compute weighted variance using bessel-like correction
        #
        w_var = np.dot(deviations ** 2, w) / (w_sum - w_sum / n)
        out[f"{prefix}_variance"] = w_var
        out[f"{prefix}_std"] = np.sqrt(w_var)

    # compute weighted skewness (require at least 3 points)
    #
    if n >= 3 and out[f"{prefix}_std"] > 0:
        deviations = v - w_mean
        m3 = np.dot(deviations ** 3, w) / w_sum
        out[f"{prefix}_skewness"] = m3 / (out[f"{prefix}_std"] ** 3)

    # compute weighted kurtosis excess (require at least 4 points)
    #
    if n >= 4 and out[f"{prefix}_std"] > 0:
        deviations = v - w_mean
        m4 = np.dot(deviations ** 4, w) / w_sum
        out[f"{prefix}_kurtosis_excess"] = m4 / (out[f"{prefix}_std"] ** 4) - 3.0

    # return the statistics dictionary
    #
    return out


def aggregate_to_daily(df_event):
    """
    function: aggregate_to_daily

    arguments:
      df_event: cleaned event-level timeline dataframe

    return:
      daily feature dataframe

    description:
      creates intervals between consecutive events slices any intervals
      that cross midnight, and aggregates to daily feature rows with time
      weighted stats.
    """

    # need at least two rows to form an interval between them
    #
    if len(df_event) < 2:
        return pd.DataFrame()

    # the interval start timestamps come from every row except the
    # last. the end timestamps come from every row except the first.
    # this pairs row[i] with row[i+1] across the full dataframe.
    #
    starts = df_event["Timestamp"].iloc[:-1].values
    ends = df_event["Timestamp"].iloc[1:].values

    # the running mode for each interval is the mode that was active
    # DURING the interval, which is Interval_RunningMode on the end
    # row (already computed as shifted running mode in earlier blocks)
    #
    modes = df_event["Interval_RunningMode"].iloc[1:].values

    # continuous state values (temperature, setpoint, outdoor weather)
    # come from the start row since they represent the measured state
    # at the beginning of the interval and persist until the next ping
    #
    temps = pd.to_numeric(
        df_event["Temperature"].iloc[:-1], errors="coerce"
    ).values
    sps = pd.to_numeric(
        df_event["Setpoint"].iloc[:-1], errors="coerce"
    ).values
    ot = pd.to_numeric(
        df_event["Outdoor_Temperature"].iloc[:-1], errors="coerce"
    ).values
    oh = pd.to_numeric(
        df_event["outsideHumidity"].iloc[:-1], errors="coerce"
    ).values

    # compute binary fan-active flag for each interval start row.
    # the fan is considered active if FanState is true/1.
    #
    fan_raw = (
        df_event["FanState"].iloc[:-1]
        .astype(str).str.strip().str.lower()
    )
    fan_active = fan_raw.isin({"true", "1"}).astype(int).values

    # assemble the full interval dataframe in one operation
    #
    intervals = pd.DataFrame({
        "start": pd.DatetimeIndex(starts),
        "end": pd.DatetimeIndex(ends),
        "RunningMode": modes,
        "Temperature": temps,
        "Setpoint": sps,
        "Outdoor_Temperature": ot,
        "outsideHumidity": oh,
        "FanState_Active": fan_active,
    })

    # drop any intervals where either timestamp is null (rare but
    # possible at the boundaries of the event stream)
    #
    intervals = intervals.dropna(subset=["start", "end"])

    # compute the calendar date (midnight) of each interval boundary.
    # if start_date == end_date the interval stays within one day.
    #
    start_dates = intervals["start"].dt.normalize()
    end_dates = intervals["end"].dt.normalize()
    same_day_mask = start_dates == end_dates

    # same-day intervals. assign date and compute
    # duration 
    #
    same = intervals[same_day_mask].copy()
    same["Date"] = same["start"].dt.date
    same["Duration_Seconds"] = (
        same["end"] - same["start"]
    ).dt.total_seconds()

    # cross-midnight intervals. loop only
    # over this small subset and split each interval at every midnight
    # boundary it crosses. after virtual expiration rows, the longest
    # possible interval is from a virtual row to the next real ping,
    # which usually crosses at most one midnight.
    #
    cross = intervals[~same_day_mask]
    cross_pieces = []
    state_cols = [
        "RunningMode", "Temperature", "Setpoint",
        "Outdoor_Temperature", "outsideHumidity", "FanState_Active"
    ]

    for _, row in cross.iterrows():
        curr = row["start"]
        end_t = row["end"]
        while curr < end_t:

            # find the next midnight after the current position
            #
            nxt_mid = curr.normalize() + pd.Timedelta(days=1)

            # the piece ends at midnight or the interval end,
            # whichever comes first
            #
            piece_end = min(end_t, nxt_mid)

            # build the slice record for this calendar-day piece
            #
            piece = {
                "Date": curr.date(),
                "Duration_Seconds": (piece_end - curr).total_seconds(),
            }
            for c in state_cols:
                piece[c] = row[c]
            cross_pieces.append(piece)

            # advance past this piece to the next day
            #
            curr = piece_end

    # combine same-day slices and cross-midnight pieces into a single
    # dataframe. this is the final midnight-sliced interval table
    # that daily aggregation will group over.
    #
    keep = ["Date", "Duration_Seconds"] + state_cols
    if cross_pieces:
        res = pd.concat(
            [same[keep], pd.DataFrame(cross_pieces)],
            ignore_index=True
        )
    else:
        res = same[keep].reset_index(drop=True)

    # add date column to the event dataframe for ping-based grouping
    #
    df_event["Date_ext"] = df_event["Timestamp"].dt.date

    # collect daily feature rows
    #
    daily_rows = []

    # iterate over each calendar date
    #
    for date, grp_slice in res.groupby("Date"):

        # get raw event pings for this date
        #
        grp_ping = df_event[df_event["Date_ext"] == date]

        # compute runtime hours by mode
        #
        m_sec = grp_slice.groupby("RunningMode")["Duration_Seconds"].sum().to_dict()
        h_hrs = m_sec.get("heat", 0) / 3600.0
        c_hrs = m_sec.get("cool", 0) / 3600.0
        o_hrs = m_sec.get("off", 0) / 3600.0
        u_hrs = m_sec.get("unknown", 0) / 3600.0

        # compute daily runtime hours (heating + cooling, no poisoning)
        #
        rt_hrs = h_hrs + c_hrs

        # compute total known-mode hours
        #
        known_hrs = h_hrs + c_hrs + o_hrs

        # compute fan runtime
        #
        fan_sec = (grp_slice["FanState_Active"] * grp_slice["Duration_Seconds"]).sum()
        fan_hrs = fan_sec / 3600.0

        # compute fan runtime ratio
        #
        fan_ratio = fan_hrs / known_hrs if known_hrs > 0 else 0.0

        # compute time-weighted means using slice durations
        #
        def calc_weighted_mean(val_col):

            # filter to slices with valid values
            #
            valid = grp_slice[grp_slice[val_col].notna()]

            # return nan if no valid duration
            #
            if valid["Duration_Seconds"].sum() == 0:
                return np.nan

            # compute weighted average
            #
            return (
                (valid[val_col] * valid["Duration_Seconds"]).sum()
                / valid["Duration_Seconds"].sum()
            )

        # compute weighted means for continuous variables
        #
        wt_temp = calc_weighted_mean("Temperature")
        wt_sp = calc_weighted_mean("Setpoint")
        wt_out_temp = calc_weighted_mean("Outdoor_Temperature")
        wt_out_hum = calc_weighted_mean("outsideHumidity")

        # count setpoint changes from the raw pings
        #
        if "Setpoint" in grp_ping:
            sp_changes = grp_ping["Setpoint"].dropna().diff().dropna().ne(0).sum()
        else:
            sp_changes = 0

        # count occupancy pings
        #
        if "Occupied" in grp_ping:
            occ_clean = grp_ping["Occupied"].astype(str).str.strip().str.lower()
            occ_true_count = (occ_clean == "true").sum()
            occ_false_count = (occ_clean == "false").sum()
        else:
            occ_true_count = 0
            occ_false_count = 0

        # build the daily feature dictionary
        #
        day_dict = {
            "Equipment_ID": df_event.iloc[0]["Equipment_ID"],
            "Date": pd.to_datetime(date),
            "daily_heating_hours": h_hrs,
            "daily_cooling_hours": c_hrs,
            "daily_off_hours": o_hrs,
            "daily_unknown_hours": u_hrs,
            "daily_runtime_hours": rt_hrs,
            "daily_fan_on_hours": fan_hrs,
            "fan_runtime_ratio": fan_ratio,
            "setpoint_change_count": sp_changes,
            "occupied_ping_count": occ_true_count,
            "unoccupied_ping_count": occ_false_count,
            "indoor_temp_time_weighted_mean": wt_temp,
            "setpoint_time_weighted_mean": wt_sp,
            "outdoor_temp_time_weighted_mean": wt_out_temp,
            "outdoor_humidity_time_weighted_mean": wt_out_hum,
        }

        # compute the setpoint-to-indoor comfort gap
        #
        if pd.notna(wt_sp) and pd.notna(wt_temp):
            day_dict["setpoint_gap_mean"] = wt_sp - wt_temp
        else:
            day_dict["setpoint_gap_mean"] = np.nan

        # add calendar features
        #
        dow = day_dict["Date"].dayofweek
        day_dict["day_of_week"] = dow
        day_dict["is_weekend"] = 1 if dow >= 5 else 0

        # add month and circular encoding
        #
        mth = day_dict["Date"].month
        day_dict["month"] = mth
        day_dict["month_sin"] = np.sin(2 * np.pi * mth / 12)
        day_dict["month_cos"] = np.cos(2 * np.pi * mth / 12)

        # compute time-weighted distributional statistics using slice data
        #
        day_dict.update(weighted_stats_for_slices(
            grp_slice["Temperature"].values,
            grp_slice["Duration_Seconds"].values,
            "indoor_temp"
        ))
        day_dict.update(weighted_stats_for_slices(
            grp_slice["Setpoint"].values,
            grp_slice["Duration_Seconds"].values,
            "setpoint"
        ))
        day_dict.update(weighted_stats_for_slices(
            grp_slice["Outdoor_Temperature"].values,
            grp_slice["Duration_Seconds"].values,
            "outdoor_temp"
        ))
        day_dict.update(weighted_stats_for_slices(
            grp_slice["outsideHumidity"].values,
            grp_slice["Duration_Seconds"].values,
            "outdoor_hum"
        ))

        # append to daily rows
        #
        daily_rows.append(day_dict)

    # convert to dataframe and sort by date
    #
    return pd.DataFrame(daily_rows).sort_values("Date").reset_index(drop=True)


def compute_outdoor_gradient(temps, window):
    """
    function: compute_outdoor_gradient

    arguments:
      temps: array-like of daily mean outdoor temperatures
      window: number of prior days to include in the linear fit

    return:
      numpy array of slope values (degrees per day)

    description:
      for each day, fits a line to the mean outdoor temp over the
      prior N days and returns the slope. positive = warming trend,
      negative = cooling trend.
    """

    # convert to numpy array
    #
    temps = np.asarray(temps, dtype=float)
    n = len(temps)

    # initialize gradient array with nans
    #
    grads = np.full(n, np.nan)

    # compute gradient for each position with enough history
    #
    for i in range(window, n):

        # extract the window of values
        #
        w = temps[i - window:i]

        # skip if too few valid values
        #
        valid = ~np.isnan(w)
        if valid.sum() < 3:
            continue

        # set up x values for the linear fit
        #
        x = np.arange(window, dtype=float)
        x_valid = x[valid]
        y_valid = w[valid]

        # compute means
        #
        x_mean = x_valid.mean()
        y_mean = y_valid.mean()

        # compute denominator for slope
        #
        denom = ((x_valid - x_mean) ** 2).sum()

        # avoid division by zero
        #
        if denom == 0:
            continue

        # compute slope (degrees per day)
        #
        grads[i] = ((x_valid - x_mean) * (y_valid - y_mean)).sum() / denom

    # return the gradient array
    #
    return grads


# compute true daily outdoor weather aggregates from the weather file
#
w_df["Date"] = pd.to_datetime(w_df["Timestamp"].dt.date)
true_weather = w_df.groupby("Date").agg(
    true_outside_min=("outsideMinTemp", "min"),
    true_outside_max=("outsideMaxTemp", "max"),
    true_outside_mean=("Outdoor_Temperature", "mean"),
    true_humidity_mean=("outsideHumidity", "mean")
).reset_index()

# sort the weather timeline chronologically
#
true_weather = true_weather.sort_values("Date").reset_index(drop=True)

# compute the outdoor temperature trend gradient on the weather timeline
#
true_weather["outdoor_temp_trend_gradient"] = compute_outdoor_gradient(
    true_weather["true_outside_mean"].values, OUTDOOR_TREND_WINDOW
)


In [ ]:
"""
==========================================================================
BLOCK 10: MASTER PIPELINE EXECUTION LOOP
==========================================================================
Compiles all preprocessing steps into a single function. Iterates
through every indoor file, processes it, merges weather aggregates
(including the outdoor gradient from Block 9), and writes daily output.
==========================================================================
"""

def sanitize_equipment_id(raw_id):
    """
    function: sanitize_equipment_id

    arguments:
      raw_id: raw equipment id string derived from filename

    return:
      cleaned string safe for filenames

    description:
      removes redundant prefixes and special characters from equipment ids.
    """

    # strip the common prefix if duplicated
    #
    clean = raw_id.replace("timeseries_table_", "", 1)

    # replace spaces and parentheses with underscores
    #
    clean = clean.replace(" ", "_").replace("(", "").replace(")", "")

    # collapse multiple underscores
    #
    while "__" in clean:
        clean = clean.replace("__", "_")

    # strip leading/trailing underscores
    #
    return clean.strip("_")


def process_one_indoor_file(path, outdoor_df, true_weather_df):
    """
    function: process_one_indoor_file

    arguments:
      path: file path to one indoor csv
      outdoor_df: processed outdoor weather dataframe
      true_weather_df: aggregated daily weather dataframe (includes gradient)

    return:
      daily feature dataframe for this equipment

    description:
      runs the full preprocessing pipeline on one indoor file.
    """

    # read and standardize the raw file
    #
    raw = pd.read_csv(path, sep=";", low_memory=False)
    raw = standardize_indoor_columns(raw)

    # parse timestamps and drop invalid rows
    #
    raw["Timestamp"] = pd.to_datetime(raw["Timestamp"], errors="coerce")
    raw = raw.dropna(subset=["Timestamp"]).sort_values("Timestamp").reset_index(drop=True)

    # add equipment id and row counter
    #
    raw["Equipment_ID"] = os.path.splitext(os.path.basename(path))[0]
    raw["raw_row_id"] = np.arange(len(raw))

    # resolve same-timestamp bursts
    #
    event = resolve_same_timestamp_bursts(raw)

    # extract and normalize running mode
    #
    event["RunningMode_raw"] = event.get("RunningMode", np.nan)
    event["RunningMode_clean"] = event["RunningMode_raw"].map(normalize_running_mode)

    # inject virtual expiration rows
    #
    event = inject_virtual_expiration_rows(event)

    # forward-fill running mode
    #
    event["RunningMode"] = event["RunningMode_clean"].ffill()

    # apply bounded interpolation to temperature
    #
    event["Temperature"] = bounded_time_interpolate(
        event["Temperature"], event["Timestamp"]
    )

    # forward-fill static metadata if present
    #
    if "RentalStatus" in event.columns:
        event["RentalStatus"] = event["RentalStatus"].ffill().bfill()

    # forward-fill each step-like column
    #
    for c in STEP_LIKE_COLS:
        if c in event.columns:

            # coerce setpoint to numeric
            #
            if c == "Setpoint":
                event[c] = pd.to_numeric(event[c], errors="coerce")

            # forward-fill
            #
            event[c] = event[c].ffill()

            # replace reset tokens with nan
            #
            if c == "Setpoint":
                event[c] = event[c].replace(NUMERIC_RESET_TOKEN, np.nan)

    # sort by timestamp and row id, then set interval mode
    #
    event = event.sort_values(
        ["Timestamp", "raw_row_id"], kind="mergesort"
    ).reset_index(drop=True)
    event["Interval_RunningMode"] = event["RunningMode"].shift(1).fillna("unknown")

    # attach outdoor weather
    #
    event = attach_weather_to_indoor(event, outdoor_df)

    # aggregate to daily features
    #
    daily = aggregate_to_daily(event)

    # merge true weather aggregates (includes outdoor_temp_trend_gradient)
    #
    daily["Date"] = pd.to_datetime(daily["Date"])
    daily = pd.merge(daily, true_weather_df, on="Date", how="left")

    # compute indoor-outdoor temperature gradient
    #
    daily["temp_gradient_mean"] = (
        daily["indoor_temp_time_weighted_mean"] - daily["true_outside_mean"]
    )

    # return the daily dataframe
    #
    return daily


# process all indoor files
#
print("\nprocessing all indoor files:")
for f in indoor_files:

    # extract the raw equipment id
    #
    uid = os.path.splitext(os.path.basename(f))[0]

    # process the file
    #
    daily = process_one_indoor_file(f, w_df, true_weather)

    # drop count_nonnull columns (diagnostic only, not modeling features)
    #
    nonnull_cols = [c for c in daily.columns if "count_nonnull" in c]
    daily = daily.drop(columns=nonnull_cols, errors="ignore")

    # generate a clean output filename
    #
    clean_name = sanitize_equipment_id(uid)
    out_path = os.path.join(OUT_DAILY, f"{clean_name}_daily.csv")

    # write to disk
    #
    daily.to_csv(out_path, index=False)

    # log progress
    #
    print(f"  processed: {clean_name}_daily.csv")


In [5]:
"""
==========================================================================
BLOCK 11: DATA RECONSTRUCTION AND FEATURE SELECTION
==========================================================================
Reloads processed daily files, enforces a continuous calendar grid,
then runs feature selection on the base (non-lagged) features using
mutual information, pearson correlation, and rfe. the mi ranking is
used to select features. features are then split into lag-worthy
(evolving state that benefits from historical context) and same-day
(calendar, weather extremes, gradient) categories.
==========================================================================
"""

# number of top features to keep based on mutual information
#
N_TOP_FEATURES = 25

# features that should NOT be lagged because they describe today only,
# are static calendar encodings, or already encode multi-day history
#
SAME_DAY_ONLY = {
    "day_of_week", "is_weekend",
    "month", "month_sin", "month_cos",
    "true_outside_min", "true_outside_max",
    "true_outside_mean", "true_humidity_mean",
    "outdoor_temp_trend_gradient",
    "setpoint_change_count",
    "occupied_ping_count", "unoccupied_ping_count",
}

# discover all processed daily csv files
#
daily_paths = sorted(glob.glob(os.path.join(OUT_DAILY, "*.csv")))

# concatenate all daily files
#
raw_master = pd.concat([pd.read_csv(f) for f in daily_paths], ignore_index=True)

# parse the date column
#
raw_master["Date"] = pd.to_datetime(raw_master["Date"])

# drop columns that would leak ping frequency information
#
leak_cols = [c for c in raw_master.columns if "count_nonnull" in c or "ping_count" in c]
raw_master = raw_master.drop(columns=leak_cols, errors="ignore")


def local_grid(grp):
    """
    function: local_grid

    arguments:
      grp: dataframe group for one equipment id

    return:
      dataframe reindexed to a continuous daily calendar

    description:
      fills in missing days with empty rows to prevent time leakage
      during lag generation.
    """

    # create a continuous daily date range
    #
    dr = pd.date_range(start=grp["Date"].min(), end=grp["Date"].max(), freq='D')

    # reindex to the continuous grid
    #
    grp = grp.set_index("Date").reindex(dr)

    # forward-fill the equipment id
    #
    grp["Equipment_ID"] = grp["Equipment_ID"].ffill().bfill()

    # return with date as a column
    #
    return grp.rename_axis("Date").reset_index()


# apply calendar gridding per equipment id
#
master_processed = raw_master.groupby(
    "Equipment_ID", group_keys=False
).apply(local_grid)

# ---------------------------------------------------------------------------
#
# feature selection on base (non-lagged) features
#
# ---------------------------------------------------------------------------

# identify all numeric columns as candidate base features
#
num_cols = master_processed.select_dtypes(include=[np.number]).columns.tolist()

# exclude target and leakage columns from the candidate set
#
candidate_feats = [
    c for c in num_cols
    if c not in LEAKAGE_COLS
]

# prepare a clean scoring dataset
#
score_df = master_processed.dropna(subset=["daily_runtime_hours"]).copy()
avail_feats = [f for f in candidate_feats if f in score_df.columns]
score_df = score_df.dropna(subset=avail_feats).reset_index(drop=True)

# extract the feature matrix and target
#
X_score = score_df[avail_feats]
y_score = score_df["daily_runtime_hours"]

# ---------------------------------------------------------------------------
#
# pearson correlation
#
# ---------------------------------------------------------------------------

# compute pearson correlation of each feature against the target
#
pearson_corr = X_score.corrwith(y_score).abs()
pearson_df = pd.DataFrame({
    "feature": pearson_corr.index,
    "pearson_abs": pearson_corr.values
}).sort_values("pearson_abs", ascending=False).reset_index(drop=True)

# display the top 30 features by pearson correlation
#
print("top 30 features by |pearson correlation| with daily_runtime_hours:")
print(pearson_df.head(30).to_string(index=False))

# ---------------------------------------------------------------------------
#
# mutual information scoring
#
# ---------------------------------------------------------------------------

# compute mutual information scores against the target
#
mi_scores = mutual_info_regression(X_score, y_score, random_state=27, n_neighbors=5)

# create a ranked dataframe
#
mi_df = pd.DataFrame({
    "feature": avail_feats,
    "mi_score": mi_scores
}).sort_values("mi_score", ascending=False).reset_index(drop=True)

# display the top 30 features by mutual information
#
print("\ntop 30 features by mutual information with daily_runtime_hours:")
print(mi_df.head(30).to_string(index=False))

# ---------------------------------------------------------------------------
#
# recursive feature elimination (rfe)
#
# ---------------------------------------------------------------------------

# instantiate a small random forest for rfe
#
rf_rfe = RandomForestRegressor(
    n_estimators=50, max_depth=5, random_state=27, n_jobs=1
)

# run rfe to rank all features
#
rfe = RFE(estimator=rf_rfe, n_features_to_select=1, step=5)
rfe.fit(X_score, y_score)

# create a ranked dataframe
#
rfe_df = pd.DataFrame({
    "feature": avail_feats,
    "rfe_rank": rfe.ranking_
}).sort_values("rfe_rank").reset_index(drop=True)

# display the top 30 features by rfe rank
#
print("\ntop 30 features by rfe rank:")
print(rfe_df.head(30).to_string(index=False))

# ---------------------------------------------------------------------------
#
# select the top N features based on mutual information
#
# ---------------------------------------------------------------------------

# take the top N features by mi score
#
selected_feats = mi_df.head(N_TOP_FEATURES)["feature"].tolist()

# always include daily_runtime_hours for target lag generation
#
if "daily_runtime_hours" not in selected_feats:
    selected_feats.append("daily_runtime_hours")

# split selected features into lag-worthy and same-day-only
#
lag_worthy_feats = [f for f in selected_feats if f not in SAME_DAY_ONLY]
same_day_feats = [f for f in selected_feats if f in SAME_DAY_ONLY]

# report selection
#
print(f"\nselected {len(selected_feats)} base features by mutual information:")
print(f"  lag-worthy ({len(lag_worthy_feats)}): features that get lag_1..lag_{LAG_DAYS}")
for feat in lag_worthy_feats:
    mi_match = mi_df.loc[mi_df["feature"] == feat, "mi_score"].values
    mi_str = f"mi={mi_match[0]:.4f}" if len(mi_match) > 0 else "target (forced)"
    print(f"    {feat}: {mi_str}")
print(f"  same-day only ({len(same_day_feats)}): no lags generated")
for feat in same_day_feats:
    mi_match = mi_df.loc[mi_df["feature"] == feat, "mi_score"].values
    mi_str = f"mi={mi_match[0]:.4f}" if len(mi_match) > 0 else "target (forced)"
    print(f"    {feat}: {mi_str}")


/tmp/ipykernel_51428/2193086223.py:85: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(local_grid)


top 30 features by |pearson correlation| with daily_runtime_hours:
                        feature  pearson_abs
                daily_off_hours     0.889365
            daily_heating_hours     0.684620
            daily_cooling_hours     0.413765
            daily_unknown_hours     0.301538
            outdoor_temp_median     0.278983
               outdoor_temp_q25     0.277688
outdoor_temp_time_weighted_mean     0.277451
              outdoor_temp_mean     0.277451
              true_outside_mean     0.276143
               true_outside_max     0.271369
               outdoor_temp_q75     0.270577
               outdoor_temp_max     0.270069
               outdoor_temp_min     0.266054
               true_outside_min     0.264139
             temp_gradient_mean     0.260710
                indoor_temp_min     0.174677
                      month_sin     0.158034
    outdoor_temp_trend_gradient     0.150681
                          month     0.148446
              setpoint_gap_mean  

In [6]:
"""
==========================================================================
BLOCK 12: LAG GENERATION ON LAG-WORTHY FEATURES
==========================================================================
Generates lag_1 through lag_7 only for features that capture evolving
state (runtime, temperatures, comfort gaps, etc). Same-day features
(calendar, weather extremes, gradient) are included as-is without lags.
==========================================================================
"""

# filter lag-worthy features to those present in the dataframe
#
lag_cols_for_shift = [c for c in lag_worthy_feats if c in master_processed.columns]

# generate lags 1..LAG_DAYS for lag-worthy features only
#
lags = []
for i in range(1, LAG_DAYS + 1):

    # shift lag-worthy columns by i days within each equipment group
    #
    s = master_processed.groupby("Equipment_ID")[lag_cols_for_shift].shift(i)

    # rename columns with lag suffix
    #
    s.columns = [f"{c}_lag_{i}" for c in s.columns]

    # append to lag list
    #
    lags.append(s)

# concatenate original data with lag features
#
master_local_lags = pd.concat([master_processed] + lags, axis=1)

# cast equipment id to categorical for model routing
#
master_local_lags["Equipment_ID"] = master_local_lags["Equipment_ID"].astype("category")

# define the lag feature set
#
lag_feats = [c for c in master_local_lags.columns if "_lag_" in c]

# define same-day features (no lags, used as-is)
#
day_only_feats = [c for c in same_day_feats if c != "daily_runtime_hours"]

# compile the final feature set: same-day exogenous + lag features only.
# current-day endogenous features (indoor temp, fan hours, setpoint gap,
# etc.) are excluded because they are concurrent outcomes of hvac
# operation that would not be available at prediction time.
#
final_feature_set = day_only_feats + lag_feats

# report feature counts
#
print(f"\nfinal feature set: {len(final_feature_set)} total")
print(f"  same-day exogenous features: {len(day_only_feats)}")
print(f"  lag features: {len(lag_feats)}")



final feature set: 146 total
  same-day exogenous features: 6
  lag features: 140


In [7]:
"""
==========================================================================
BLOCK 13: STRATIFIED BLOCKED TIME SERIES SPLIT
==========================================================================
Divides the calendar into 14-day contiguous blocks and assigns each
block to a meteorological season. Within each season, allocates the
last 20% of blocks to testing. The same calendar blocks are used for
all households, preventing temporal leakage through shared outdoor
weather features while ensuring all seasons appear in both sets.
==========================================================================
"""

# define block size in days
#
BLOCK_SIZE_DAYS = 14


def month_to_season(m):
    """
    function: month_to_season

    arguments:
      m: integer month (1-12)

    return:
      string season name

    description:
      maps a calendar month to its meteorological season.
    """

    # winter: december, january, february
    #
    if m in {12, 1, 2}:
        return "winter"

    # spring: march, april, may
    #
    if m in {3, 4, 5}:
        return "spring"

    # summer: june, july, august
    #
    if m in {6, 7, 8}:
        return "summer"

    # fall: september, october, november
    #
    return "fall"


# determine the global date range across all households
#
all_dates = master_local_lags["Date"].dropna()
date_min = all_dates.min().normalize()
date_max = all_dates.max().normalize()

# create contiguous 14-day blocks spanning the full range
#
block_starts = pd.date_range(start=date_min, end=date_max, freq=f"{BLOCK_SIZE_DAYS}D")
blocks = []
for i, bs in enumerate(block_starts):

    # compute block end and midpoint
    #
    be = bs + pd.Timedelta(days=BLOCK_SIZE_DAYS - 1)
    mid = bs + pd.Timedelta(days=BLOCK_SIZE_DAYS // 2)

    # assign season based on midpoint month
    #
    blocks.append({
        "block_id": i,
        "start": bs,
        "end": be,
        "season": month_to_season(mid.month)
    })

# convert to dataframe
#
block_df = pd.DataFrame(blocks)

# within each season, assign the last ~20% of blocks to test
#
block_df["split"] = "train"
for season in block_df["season"].unique():

    # isolate blocks for this season in chronological order
    #
    season_idx = block_df[block_df["season"] == season].sort_values("block_id").index
    n_test = max(1, int(np.ceil(len(season_idx) * 0.2)))

    # assign the last n_test blocks to test
    #
    block_df.loc[season_idx[-n_test:], "split"] = "test"

# map each day in the master dataframe to its block split assignment
#
master_local_lags["split"] = "train"
for _, blk in block_df.iterrows():

    # find rows within this block date range
    #
    day_mask = (
        (master_local_lags["Date"] >= blk["start"]) &
        (master_local_lags["Date"] <= blk["end"])
    )

    # assign the block split value
    #
    master_local_lags.loc[day_mask, "split"] = blk["split"]

# create boolean masks for model training
#
train_mask = master_local_lags["split"] == "train"
test_mask = master_local_lags["split"] == "test"

# report split statistics
#
print(f"\nstratified blocked split ({BLOCK_SIZE_DAYS}-day blocks):")
print(f"  total blocks: {len(block_df)}")
for season in ["winter", "spring", "summer", "fall"]:
    s_blk = block_df[block_df["season"] == season]
    n_tr = (s_blk["split"] == "train").sum()
    n_te = (s_blk["split"] == "test").sum()
    print(f"  {season}: {n_tr} train, {n_te} test")
print(f"  train rows: {train_mask.sum()}, test rows: {test_mask.sum()}")


stratified blocked split (14-day blocks):
  total blocks: 53
  winter: 9 train, 3 test
  spring: 12 train, 3 test
  summer: 9 train, 3 test
  fall: 11 train, 3 test
  train rows: 26459, test rows: 15333


In [8]:
"""
==========================================================================
BLOCK 14: TEMPORAL CLUSTERING
==========================================================================
Clusters households by daily runtime patterns using tslearn
TimeSeriesKMeans with DTW metric. Assigns each household to a
behavioral cluster for downstream cluster-based modeling.
==========================================================================
"""

try:
    from tslearn.clustering import TimeSeriesKMeans
    from tslearn.preprocessing import TimeSeriesScalerMeanVariance
    TSLEARN_AVAILABLE = True
except ImportError:
    TSLEARN_AVAILABLE = False
    print("tslearn not installed. install with: pip install tslearn")
    print("skipping temporal clustering.")

if TSLEARN_AVAILABLE:

    # define the number of clusters
    #
    N_CLUSTERS = 3

    # pivot daily runtime into a matrix of (houses x days)
    #
    pivot_df = master_local_lags.pivot_table(
        index="Equipment_ID", columns="Date",
        values="daily_runtime_hours", aggfunc="first"
    )

    # fill missing days with 0 for clustering
    #
    pivot_filled = pivot_df.fillna(0).values

    # reshape for tslearn: (n_samples, n_timestamps, n_features)
    #
    ts_data = pivot_filled.reshape(pivot_filled.shape[0], pivot_filled.shape[1], 1)

    # scale the time series
    #
    scaler_ts = TimeSeriesScalerMeanVariance()
    ts_scaled = scaler_ts.fit_transform(ts_data)

    # replace any nans from scaling with 0
    #
    ts_scaled = np.nan_to_num(ts_scaled, nan=0.0)

    # fit the time series k-means model with dtw metric
    #
    km = TimeSeriesKMeans(
        n_clusters=N_CLUSTERS,
        metric="dtw",
        max_iter=30,
        random_state=27,
        verbose=0
    )
    cluster_labels = km.fit_predict(ts_scaled)

    # map cluster labels back to equipment ids
    #
    cluster_map = dict(zip(pivot_df.index, cluster_labels))

    # add cluster assignment to the master dataframe
    #
    master_local_lags["cluster"] = master_local_lags["Equipment_ID"].map(cluster_map)

    # report cluster sizes
    #
    print(f"\ntemporal clustering ({N_CLUSTERS} clusters, dtw metric):")
    for c_id in range(N_CLUSTERS):
        n_houses = sum(1 for v in cluster_map.values() if v == c_id)
        print(f"  cluster {c_id}: {n_houses} households")


/tmp/ipykernel_51428/537390315.py:28: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_df = master_local_lags.pivot_table(



temporal clustering (3 clusters, dtw metric):
  cluster 0: 26 households
  cluster 1: 33 households
  cluster 2: 41 households


In [9]:
"""
==========================================================================
BLOCK 15: PER-HOUSE MODELS
==========================================================================
Trains one LightGBM model per household using the stratified blocked
split from Block 13. Pools predictions across all houses for global
performance metrics.
==========================================================================
"""

from sklearn.metrics import root_mean_squared_error

# define per-house model parameters
#
local_params = {
    'n_estimators': 300,
    'learning_rate': 0.05,
    'num_leaves': 15,
    'max_depth': 5,
    'min_child_samples': 10,
    'random_state': 27,
    'n_jobs': -1,
    'verbose': -1
}

# initialize pooled tracking arrays
#
pooled_y_true = []
pooled_y_pred = []
successful_models = 0
flagged_houses = []

# iterate over each equipment id
#
for house_id in master_local_lags["Equipment_ID"].unique():

    # filter to this house
    #
    house_df = master_local_lags[
        master_local_lags["Equipment_ID"] == house_id
    ].copy()
    house_df = house_df.dropna(subset=["daily_runtime_hours"])

    # apply the stratified blocked split
    #
    h_train = house_df[house_df["split"] == "train"]
    h_test = house_df[house_df["split"] == "test"]

    # skip houses with insufficient train or test data
    #
    if len(h_train) < 40 or len(h_test) < 10:
        continue

    # split features and target
    #
    X_train_local = h_train[final_feature_set]
    y_train_local = h_train["daily_runtime_hours"]
    X_test_local = h_test[final_feature_set]
    y_test_local = h_test["daily_runtime_hours"]

    # train the model
    #
    local_model = LGBMRegressor(**local_params)
    local_model.fit(X_train_local, y_train_local)

    # predict and evaluate
    #
    preds = local_model.predict(X_test_local)
    local_r2 = r2_score(y_test_local, preds)

    # flag houses with negative r2
    #
    if local_r2 < 0.0:
        flagged_houses.append(house_id)

    # pool predictions
    #
    pooled_y_true.extend(y_test_local.values)
    pooled_y_pred.extend(preds)
    successful_models += 1

# compute pooled metrics
#
pooled_rmse = root_mean_squared_error(pooled_y_true, pooled_y_pred)
pooled_mae = mean_absolute_error(pooled_y_true, pooled_y_pred)
pooled_r2 = r2_score(pooled_y_true, pooled_y_pred)

# compute adjusted r-squared
#
n_pool = len(pooled_y_true)
p_pool = len(final_feature_set)
pooled_adj_r2 = 1 - (1 - pooled_r2) * (n_pool - 1) / (n_pool - p_pool - 1)

# report results
#
print(f"\nper-house results (stratified blocked split):")
print(f"  models trained: {successful_models}")
print(f"  flagged (r2 < 0): {len(flagged_houses)}")
print(f"  pooled rmse:   {pooled_rmse:.4f} hours")
print(f"  pooled mae:    {pooled_mae:.4f} hours")
print(f"  pooled r2:     {pooled_r2:.4f}")
print(f"  pooled adj_r2: {pooled_adj_r2:.4f}")



per-house results (stratified blocked split):
  models trained: 100
  flagged (r2 < 0): 3
  pooled rmse:   3.3889 hours
  pooled mae:    2.3371 hours
  pooled r2:     0.6482
  pooled adj_r2: 0.6448


In [32]:
"""
==========================================================================
BLOCK 16: GLOBAL MODEL
==========================================================================
Trains a single LightGBM model on all households using the stratified
blocked split from Block 13.
==========================================================================
"""

# prepare global dataset
#
global_df = master_local_lags.dropna(subset=["daily_runtime_hours"]).copy()

# apply the stratified blocked split
#
g_train = global_df[global_df["split"] == "train"]
g_test = global_df[global_df["split"] == "test"]

# report split sizes
#
print(f"global split: {len(g_train)} train, {len(g_test)} test")

# define global model parameters
#
global_params = {
    'n_estimators': 800,
    'learning_rate': 0.01,
    'num_leaves': 63,
    'max_depth': 8,
    'min_child_samples': 65,
    'colsample_bytree': 0.8,
    'random_state': 27,
    'n_jobs': -1,
    'verbose': -1
}

# split features and target
#
X_train_global = g_train[final_feature_set]
y_train_global = g_train["daily_runtime_hours"]
X_test_global = g_test[final_feature_set]
y_test_global = g_test["daily_runtime_hours"]

# train the global model with early stopping
#
global_model = LGBMRegressor(**global_params)
global_model.fit(
    X_train_global, y_train_global,
    eval_set=[(X_test_global, y_test_global)],
    eval_metric="rmse",
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=0)
    ]
)

# report how many trees were actually used
#
print(f"  early stopping: used {global_model.best_iteration_} of {global_params['n_estimators']} trees")

# predict and evaluate
#
global_preds = global_model.predict(X_test_global)
g_rmse = root_mean_squared_error(y_test_global, global_preds)
g_mae = mean_absolute_error(y_test_global, global_preds)
g_r2 = r2_score(y_test_global, global_preds)

# compute adjusted r-squared
#
n_g = len(y_test_global)
p_g = len(final_feature_set)
g_adj_r2 = 1 - (1 - g_r2) * (n_g - 1) / (n_g - p_g - 1)

# report results
#
print(f"\nglobal model results (stratified blocked split):")
print(f"  rmse:   {g_rmse:.4f} hours")
print(f"  mae:    {g_mae:.4f} hours")
print(f"  r2:     {g_r2:.4f}")
print(f"  adj_r2: {g_adj_r2:.4f}")

global split: 26459 train, 15333 test
  early stopping: used 751 of 800 trees

global model results (stratified blocked split):
  rmse:   2.6675 hours
  mae:    1.8362 hours
  r2:     0.7820
  adj_r2: 0.7800


In [11]:
"""
==========================================================================
BLOCK 17: CLUSTER-BASED MODELS
==========================================================================
Trains one LightGBM model per temporal cluster from Block 14 using the
stratified blocked split from Block 13.
==========================================================================
"""

if TSLEARN_AVAILABLE:

    # initialize pooled tracking
    #
    cluster_pooled_true = []
    cluster_pooled_pred = []

    # train one model per cluster
    #
    for c_id in range(N_CLUSTERS):

        # filter to this cluster
        #
        c_df = master_local_lags[
            master_local_lags["cluster"] == c_id
        ].dropna(subset=["daily_runtime_hours"]).copy()

        # apply the stratified blocked split
        #
        c_train = c_df[c_df["split"] == "train"]
        c_test = c_df[c_df["split"] == "test"]

        # skip if insufficient data
        #
        if len(c_train) < 40 or len(c_test) < 10:
            print(f"  cluster {c_id}: insufficient data, skipping")
            continue

        # split features and target
        #
        X_train_c = c_train[final_feature_set]
        y_train_c = c_train["daily_runtime_hours"]
        X_test_c = c_test[final_feature_set]
        y_test_c = c_test["daily_runtime_hours"]

        # train model for this cluster
        #
        cluster_model = LGBMRegressor(**global_params)
        cluster_model.fit(X_train_c, y_train_c)

        # predict and pool
        #
        c_preds = cluster_model.predict(X_test_c)
        cluster_pooled_true.extend(y_test_c.values)
        cluster_pooled_pred.extend(c_preds)

        # report per-cluster metrics
        #
        c_r2 = r2_score(y_test_c, c_preds)
        c_rmse = root_mean_squared_error(y_test_c, c_preds)
        n_c = len(y_test_c)
        p_c = len(final_feature_set)
        c_adj_r2 = 1 - (1 - c_r2) * (n_c - 1) / (n_c - p_c - 1)
        print(f"  cluster {c_id}: rmse={c_rmse:.4f}, r2={c_r2:.4f}, adj_r2={c_adj_r2:.4f}")

    # compute pooled cluster metrics
    #
    if cluster_pooled_true:
        cl_rmse = root_mean_squared_error(cluster_pooled_true, cluster_pooled_pred)
        cl_mae = mean_absolute_error(cluster_pooled_true, cluster_pooled_pred)
        cl_r2 = r2_score(cluster_pooled_true, cluster_pooled_pred)
        n_cl = len(cluster_pooled_true)
        p_cl = len(final_feature_set)
        cl_adj_r2 = 1 - (1 - cl_r2) * (n_cl - 1) / (n_cl - p_cl - 1)
        print(f"\ncluster-based pooled results (stratified blocked split):")
        print(f"  rmse:   {cl_rmse:.4f} hours")
        print(f"  mae:    {cl_mae:.4f} hours")
        print(f"  r2:     {cl_r2:.4f}")
        print(f"  adj_r2: {cl_adj_r2:.4f}")


  cluster 0: rmse=2.7990, r2=0.7796, adj_r2=0.7708
  cluster 1: rmse=2.7224, r2=0.7535, adj_r2=0.7464
  cluster 2: rmse=2.7809, r2=0.7533, adj_r2=0.7475

cluster-based pooled results (stratified blocked split):
  rmse:   2.7658 hours
  mae:    1.9122 hours
  r2:     0.7657
  adj_r2: 0.7634


In [15]:
"""
==========================================================================
BLOCK 18: GLOBAL CATBOOST MODEL
==========================================================================
Trains a CatBoost regressor on all households using the stratified
blocked split from Block 13. CatBoost uses ordered boosting to reduce
prediction shift and handles NaN values natively without imputation.
==========================================================================
"""

from catboost import CatBoostRegressor

# prepare the dataset
#
cb_df = master_local_lags.dropna(subset=["daily_runtime_hours"]).copy()

# apply the stratified blocked split
#
cb_train = cb_df[cb_df["split"] == "train"]
cb_test = cb_df[cb_df["split"] == "test"]

# report split sizes
#
print(f"catboost split: {len(cb_train)} train, {len(cb_test)} test")

# define catboost parameters
#
cb_params = {
    'iterations': 800,
    'learning_rate': 0.01,
    'depth': 8,
    'l2_leaf_reg': 1.0,
    'min_data_in_leaf': 50,
    'rsm': 0.8,
    'random_seed': 27,
    'verbose': 0,
    'allow_writing_files': False
}

# split features and target
#
X_train_cb = cb_train[final_feature_set]
y_train_cb = cb_train["daily_runtime_hours"]
X_test_cb = cb_test[final_feature_set]
y_test_cb = cb_test["daily_runtime_hours"]

# train catboost with early stopping
#
cb_model = CatBoostRegressor(**cb_params)
cb_model.fit(
    X_train_cb, y_train_cb,
    eval_set=(X_test_cb, y_test_cb),
    early_stopping_rounds=50,
    verbose=0
)

# report how many trees were actually used
#
print(f"  early stopping: used {cb_model.best_iteration_} of {cb_params['iterations']} trees")

# predict and evaluate
#
cb_preds = cb_model.predict(X_test_cb)
cb_rmse = root_mean_squared_error(y_test_cb, cb_preds)
cb_mae = mean_absolute_error(y_test_cb, cb_preds)
cb_r2 = r2_score(y_test_cb, cb_preds)

# compute adjusted r-squared
#
n_cb = len(y_test_cb)
p_cb = len(final_feature_set)
cb_adj_r2 = 1 - (1 - cb_r2) * (n_cb - 1) / (n_cb - p_cb - 1)

# report results
#
print(f"\ncatboost results (stratified blocked split):")
print(f"  rmse:   {cb_rmse:.4f} hours")
print(f"  mae:    {cb_mae:.4f} hours")
print(f"  r2:     {cb_r2:.4f}")
print(f"  adj_r2: {cb_adj_r2:.4f}")

# show top 15 features by importance
#
cb_importance = pd.DataFrame({
    'feature': final_feature_set,
    'importance': cb_model.feature_importances_
}).sort_values('importance', ascending=False)
print(f"\ntop 15 catboost features:")
for _, row in cb_importance.head(15).iterrows():
    print(f"  {row['feature']:40s} {row['importance']:.2f}")


catboost split: 26459 train, 15333 test
  early stopping: used 799 of 800 trees

catboost results (stratified blocked split):
  rmse:   2.6925 hours
  mae:    1.8730 hours
  r2:     0.7780
  adj_r2: 0.7758

top 15 catboost features:
  daily_runtime_hours_lag_1                36.42
  daily_off_hours_lag_1                    9.82
  true_outside_mean                        7.21
  true_outside_max                         4.83
  daily_runtime_hours_lag_2                2.96
  true_outside_min                         2.47
  daily_runtime_hours_lag_3                2.05
  daily_cooling_hours_lag_1                2.05
  daily_runtime_hours_lag_7                1.76
  daily_runtime_hours_lag_4                1.44
  daily_runtime_hours_lag_5                1.02
  daily_runtime_hours_lag_6                0.99
  daily_off_hours_lag_2                    0.92
  daily_unknown_hours_lag_1                0.89
  daily_heating_hours_lag_1                0.77
